# Qwen3-8B QLoRA SFT — Chess Logic GPT

Fine-tunes Qwen3-8B with 4-bit QLoRA on chess reasoning + formal logic data.

**Setup:** Just set the runtime to T4 GPU and run all cells. Data downloads automatically from HuggingFace.

**Hardware:** T4 GPU (15GB) — fits 8B model in 4-bit (~6GB weights + optimizer).

In [ ]:
#@title 1. Install dependencies + verify GPU
!pip install -q --upgrade-strategy only-if-needed \
    torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 \
    transformers>=4.51 peft>=0.12 accelerate>=0.33 \
    datasets>=2.20 bitsandbytes python-chess orjson pyyaml zstandard huggingface_hub

import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    raise RuntimeError('No GPU! Runtime > Change runtime type > T4 GPU')

In [ ]:
#@title 2. Download training data from HuggingFace
from huggingface_hub import hf_hub_download
import os, shutil

LOCAL_DATA = '/content/data/processed'
os.makedirs(LOCAL_DATA, exist_ok=True)

REPO = 'Tobiasd2/chess-logic-gpt-sft-data'
for name in ('train_mix.jsonl', 'eval_mix.jsonl', 'eval_puzzles_ood.jsonl'):
    dst = f'{LOCAL_DATA}/{name}'
    if not os.path.exists(dst):
        print(f'Downloading {name}...')
        hf_hub_download(repo_id=REPO, filename=name, repo_type='dataset', local_dir=LOCAL_DATA)
    else:
        print(f'Already have {name}')

print('Data ready:', os.listdir(LOCAL_DATA))

In [ ]:
#@title 3. Clone repo & install package
import os, shutil

REPO_DIR = '/content/chess-logic-gpt'
if not os.path.isdir(f'{REPO_DIR}/src'):
    !git clone https://github.com/tdickers22-lgtm/chess-logic-gpt.git {REPO_DIR}

!pip install -q --no-deps -e {REPO_DIR}
print('Package installed')

In [ ]:
#@title 4. Build config + restore checkpoint from Drive (optional)
import yaml, os
from transformers.trainer_utils import get_last_checkpoint

OUTPUT_DIR = '/content/outputs/qwen3-8b-chess-logic-lora'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Optional: mount Drive to resume from a saved checkpoint
CKPT_DRIVE = None
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    CKPT_DRIVE = '/content/drive/MyDrive/chess-logic-gpt/checkpoints'
    if os.path.isdir(CKPT_DRIVE):
        drive_ckpt = get_last_checkpoint(CKPT_DRIVE)
        if drive_ckpt:
            ckpt_name = os.path.basename(drive_ckpt)
            local_ckpt = f'{OUTPUT_DIR}/{ckpt_name}'
            if not os.path.isdir(local_ckpt):
                shutil.copytree(drive_ckpt, local_ckpt)
                print(f'Restored from Drive: {ckpt_name}')
            else:
                print(f'Already local: {ckpt_name}')
        else:
            print('No checkpoint on Drive — starting fresh')
    else:
        print('No checkpoint folder on Drive — starting fresh')
except Exception as e:
    print(f'Drive not available ({e}) — starting fresh')

config = {
    'model': {
        'base_model': 'Qwen/Qwen3-8B',
        'trust_remote_code': True,
        'load_in_4bit': True,
        'enable_thinking': False,
    },
    'data': {
        'train_file': '/content/data/processed/train_mix.jsonl',
        'eval_file': '/content/data/processed/eval_mix.jsonl',
        'text_field': 'text',
        'max_seq_length': 2048,
    },
    'lora': {
        'r': 32,
        'alpha': 64,
        'dropout': 0.05,
        'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                           'gate_proj', 'up_proj', 'down_proj'],
    },
    'training': {
        'output_dir': OUTPUT_DIR,
        'num_train_epochs': 1,
        'per_device_train_batch_size': 1,
        'per_device_eval_batch_size': 1,
        'gradient_accumulation_steps': 16,
        'learning_rate': 8e-5,
        'warmup_ratio': 0.03,
        'weight_decay': 0.01,
        'logging_steps': 10,
        'eval_steps': 200,
        'save_steps': 200,
        'save_total_limit': 2,
        'bf16': True,
        'gradient_checkpointing': True,
        'resume_from_checkpoint': True,
        'report_to': 'none',
    },
}

with open('/content/train_config.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)
print('Config ready')

In [ ]:
#@title 5. Train
import time
START = time.time()

!cd /content/chess-logic-gpt && python scripts/train_lora.py --config /content/train_config.yaml

elapsed = (time.time() - START) / 3600
print(f'\nTraining took {elapsed:.2f} hours')

In [ ]:
#@title 6. Save checkpoint to Drive
import glob, shutil, os

OUTPUT_DIR = '/content/outputs/qwen3-8b-chess-logic-lora'
ckpts = sorted(glob.glob(f'{OUTPUT_DIR}/checkpoint-*'))

if ckpts and CKPT_DRIVE:
    os.makedirs(CKPT_DRIVE, exist_ok=True)
    latest = ckpts[-1]
    ckpt_name = os.path.basename(latest)
    drive_dst = f'{CKPT_DRIVE}/{ckpt_name}'
    if os.path.exists(drive_dst):
        shutil.rmtree(drive_dst)
    shutil.copytree(latest, drive_dst)
    print(f'Saved to Drive: {ckpt_name}')
elif ckpts:
    print(f'Checkpoint available at {ckpts[-1]} (Drive not mounted, skipping upload)')
else:
    print('No checkpoints found')

print('Done! Re-run this notebook to continue training.')